In [2]:
import pandas as pd

df = pd.read_csv("NFHS-5_Districts.csv")

print(df.head())
print(df.shape)

                        state state_code district  indicator_no  \
0  Andaman and Nicobar Island         AN  Nicobar             1   
1  Andaman and Nicobar Island         AN  Nicobar             2   
2  Andaman and Nicobar Island         AN  Nicobar             3   
3  Andaman and Nicobar Island         AN  Nicobar             4   
4  Andaman and Nicobar Island         AN  Nicobar             5   

                                           indicator  nfhs5_value  \
0  1. Female population age 6 years and above who...         78.0   
1               2. Population below age 15 years (%)         23.0   
2  3. Sex ratio of the total population (females ...        973.0   
3  4. Sex ratio at birth for children born in the...        927.0   
4  5. Children under age 5 years whose birth was ...         98.0   

   nfhs4_value  change_nfhs4_to_nfhs5  
0         77.2                    0.8  
1         25.4                   -2.4  
2        957.0                   16.0  
3       1060.0        

In [3]:
health_df = df.pivot_table(
    index=["state", "district"],
    columns="indicator",
    values="nfhs5_value"
).reset_index()

print("Shape:", health_df.shape)
health_df.head()

Shape: (341, 106)


indicator,state,district,1. Female population age 6 years and above who ever attended school (%),10. Households using clean fuel for cooking3 (%),100. Ever undergone an oral cavity examination for oral cancer (%),101. Women age 15 years and above who use any kind of tobacco (%),102. Men age 15 years and above who use any kind of tobacco (%),103. Women age 15 years and above who consume alcohol (%),104. Men age 15 years and above who consume alcohol (%),11. Households using iodized salt (%),...,90. Blood sugar level - very high (>160 mg/dl)23 (%),91. Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level23 (%),92. Mildly elevated blood pressure (Systolic 140-159 mm of Hg and/or Diastolic 90-99 mm of Hg) (%),93. Moderately or severely elevated blood pressure (Systolic ≥160mm of Hg and/or Diastolic ≥100mm of Hg) (%),94. Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%),95. Mildly elevated blood pressure (Systolic 140-159 mm of Hg and/or Diastolic 90-99 mm of Hg) (%),96. Moderately or severely elevated blood pressure (Systolic ≥160mm of Hg and/or Diastolic ≥100mm of Hg) (%),97. Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%),98. Ever undergone a screening test for cervical cancer (%),99. Ever undergone a breast examination for breast cancer (%)
0,Andaman and Nicobar Island,Nicobar,78.0,56.9,5.4,63.5,76.8,29.6,64.5,99.4,...,4.4,15.4,23.2,8.5,35.4,32.9,11.1,47.0,13.4,13.2
1,Andaman and Nicobar Island,North & Middle Andaman,82.7,61.3,15.8,46.8,70.5,5.1,45.3,99.9,...,6.9,18.3,18.4,4.0,27.4,22.6,6.0,32.2,1.7,0.3
2,Andaman and Nicobar Island,South Andaman,84.7,91.9,8.0,19.6,50.8,1.7,32.8,99.7,...,7.8,18.1,12.7,4.9,23.0,17.9,6.1,26.9,1.3,0.7
3,Andhra Pradesh,Anantapur,59.5,86.4,4.0,5.0,20.5,0.6,16.8,88.6,...,8.3,16.8,12.4,8.5,23.4,16.2,7.7,26.7,1.0,0.4
4,Andhra Pradesh,Chittoor,65.6,86.6,4.0,4.3,18.7,0.4,18.3,85.5,...,12.3,22.4,10.7,5.4,22.7,16.8,5.9,27.1,2.2,0.3


In [4]:
print(health_df.shape)

(341, 106)


In [5]:
health_df = health_df.fillna(health_df.mean(numeric_only=True))

print("Missing values left:", health_df.isnull().sum().sum())

Missing values left: 0


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Keep only numeric columns
X = health_df.select_dtypes(include="number")

# Scale the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train the AI model
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)

health_df["Health_Cluster"] = kmeans.fit_predict(X_scaled)

print("AI model trained successfully!")
health_df[["state","district","Health_Cluster"]].head(10)

AI model trained successfully!


/tmp/ipykernel_1220/4287795260.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  health_df["Health_Cluster"] = kmeans.fit_predict(X_scaled)


indicator,state,district,Health_Cluster
0,Andaman and Nicobar Island,Nicobar,1
1,Andaman and Nicobar Island,North & Middle Andaman,1
2,Andaman and Nicobar Island,South Andaman,1
3,Andhra Pradesh,Anantapur,1
4,Andhra Pradesh,Chittoor,1
5,Andhra Pradesh,East Godavari,1
6,Andhra Pradesh,Guntur,1
7,Andhra Pradesh,Krishna,1
8,Andhra Pradesh,Kurnool,1
9,Andhra Pradesh,Prakasam,1


In [7]:
health_df["Health_Cluster"].value_counts()


,count
Health_Cluster,
3,121
1,98
2,72
0,50


In [8]:
cluster_names = {
    0: "Low Risk",
    1: "Moderate Risk",
    2: "High Risk",
    3: "Critical"
}

health_df["Health_Risk"] = health_df["Health_Cluster"].map(cluster_names)

health_df[["district", "Health_Cluster", "Health_Risk"]].head(10)

/tmp/ipykernel_1220/4292049564.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  health_df["Health_Risk"] = health_df["Health_Cluster"].map(cluster_names)


indicator,district,Health_Cluster,Health_Risk
0,Nicobar,1,Moderate Risk
1,North & Middle Andaman,1,Moderate Risk
2,South Andaman,1,Moderate Risk
3,Anantapur,1,Moderate Risk
4,Chittoor,1,Moderate Risk
5,East Godavari,1,Moderate Risk
6,Guntur,1,Moderate Risk
7,Krishna,1,Moderate Risk
8,Kurnool,1,Moderate Risk
9,Prakasam,1,Moderate Risk


In [9]:
health_df.groupby("Health_Cluster").mean(numeric_only=True)

indicator,1. Female population age 6 years and above who ever attended school (%),10. Households using clean fuel for cooking3 (%),100. Ever undergone an oral cavity examination for oral cancer (%),101. Women age 15 years and above who use any kind of tobacco (%),102. Men age 15 years and above who use any kind of tobacco (%),103. Women age 15 years and above who consume alcohol (%),104. Men age 15 years and above who consume alcohol (%),11. Households using iodized salt (%),12. Households with any usual member covered under a health insurance/financing scheme (%),13. Children age 5 years who attended pre-primary school during the school year 2019-20 (%),...,90. Blood sugar level - very high (>160 mg/dl)23 (%),91. Blood sugar level - high or very high (>140 mg/dl) or taking medicine to control blood sugar level23 (%),92. Mildly elevated blood pressure (Systolic 140-159 mm of Hg and/or Diastolic 90-99 mm of Hg) (%),93. Moderately or severely elevated blood pressure (Systolic ≥160mm of Hg and/or Diastolic ≥100mm of Hg) (%),94. Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%),95. Mildly elevated blood pressure (Systolic 140-159 mm of Hg and/or Diastolic 90-99 mm of Hg) (%),96. Moderately or severely elevated blood pressure (Systolic ≥160mm of Hg and/or Diastolic ≥100mm of Hg) (%),97. Elevated blood pressure (Systolic ≥140 mm of Hg and/or Diastolic ≥90 mm of Hg) or taking medicine to control blood pressure (%),98. Ever undergone a screening test for cervical cancer (%),99. Ever undergone a breast examination for breast cancer (%)
Health_Cluster,,,,,,,,,,,,,,,,,,,,,
0,61.762000,39.654000,0.286000,7.178453,46.138000,1.292000,14.974000,91.622000,16.142000,14.014000,...,6.682000,15.568000,9.236000,3.866000,16.490000,11.702000,4.226000,18.702000,0.726000,0.336000
1,74.148980,82.083673,2.351020,8.334694,26.400000,3.923469,29.965306,95.282653,46.756122,19.156122,...,9.884694,19.625510,15.013265,6.784694,27.827551,19.064286,7.693878,31.330612,2.870408,1.069388
2,81.606944,42.181944,0.426389,31.376389,57.143056,4.511111,28.150000,97.448611,46.576389,11.770833,...,5.565278,14.511111,11.400000,5.004167,19.272222,15.493056,5.600000,23.477778,1.093056,0.481944
3,72.864463,60.726446,0.349587,9.158678,41.812397,1.461983,16.277686,95.684298,31.288430,11.234711,...,5.959504,14.158678,11.610744,4.453719,20.055372,13.307438,4.313223,20.408639,0.647934,0.242975


In [10]:
cluster_names = {
    0: "High Risk",
    1: "Low Risk",
    2: "Moderate Risk",
    3: "Elevated Risk"
}

health_df["Health_Risk"] = health_df["Health_Cluster"].map(cluster_names)

health_df[["district","Health_Cluster","Health_Risk"]].head(10)

indicator,district,Health_Cluster,Health_Risk
0,Nicobar,1,Low Risk
1,North & Middle Andaman,1,Low Risk
2,South Andaman,1,Low Risk
3,Anantapur,1,Low Risk
4,Chittoor,1,Low Risk
5,East Godavari,1,Low Risk
6,Guntur,1,Low Risk
7,Krishna,1,Low Risk
8,Kurnool,1,Low Risk
9,Prakasam,1,Low Risk


In [11]:
import ipywidgets as widgets
from IPython.display import display

district_dropdown = widgets.Dropdown(
    options=sorted(health_df["district"].unique()),
    description="District:",
    layout=widgets.Layout(width="500px")
)

button = widgets.Button(
    description="Analyze",
    button_style="success"
)

output = widgets.Output()

def analyze(b):

    output.clear_output()

    district = district_dropdown.value

    row = health_df[health_df["district"] == district].iloc[0]

    with output:

        print("="*55)
        print("      AI-BASED DISTRICT HEALTH PROFILING SYSTEM")
        print("="*55)

        print(f"\nDistrict       : {row['district']}")
        print(f"State          : {row['state']}")
        print(f"Health Risk    : {row['Health_Risk']}")
        print(f"Cluster Number : {row['Health_Cluster']}")
        print(f"Health Score   : {row['Health_Score']:.1f}/100")

        print("\nMost Similar Districts\n")

        similar = health_df[
            (health_df["Health_Cluster"] == row["Health_Cluster"]) &
            (health_df["district"] != district)
        ][["district","state"]].head(5)

        for _, r in similar.iterrows():
            print(f"• {r['district']} ({r['state']})")

button.on_click(analyze)

display(district_dropdown)
display(button)
display(output)

Dropdown(description='District:', layout=Layout(width='500px'), options=('Adilabad', 'Ahmedabad', 'Ahmednagar'…

Button(button_style='success', description='Analyze', style=ButtonStyle())

Output()

In [12]:
from sklearn.metrics import pairwise_distances

# Calculate distance from each district to all cluster centers
distances = pairwise_distances(X_scaled, kmeans.cluster_centers_)

# Distance to the district's own cluster center
assigned_distance = distances[
    range(len(X_scaled)),
    health_df["Health_Cluster"]
]

# Convert to a score between 0 and 100
health_df["Health_Score"] = (
    100 * (1 - assigned_distance / assigned_distance.max())
).round(1)

print("Health Score created successfully!")
health_df[["district", "Health_Score"]].head()

Health Score created successfully!


/tmp/ipykernel_1220/3701451117.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  health_df["Health_Score"] = (


indicator,district,Health_Score
0,Nicobar,0.0
1,North & Middle Andaman,33.7
2,South Andaman,51.5
3,Anantapur,58.8
4,Chittoor,59.4


In [13]:
from sklearn.metrics import pairwise_distances

distances = pairwise_distances(X_scaled, kmeans.cluster_centers_)

assigned_distance = distances[
    range(len(X_scaled)),
    health_df["Health_Cluster"]
]

health_df["Health_Score"] = (
    100 * (1 - assigned_distance / assigned_distance.max())
).round(1)

print("Health Score created successfully!")

Health Score created successfully!


In [14]:
!pip install gradio

In [15]:
import gradio as gr

def analyze_district(district):

    row = health_df[health_df["district"] == district].iloc[0]

    similar = health_df[
        (health_df["Health_Cluster"] == row["Health_Cluster"]) &
        (health_df["district"] != district)
    ][["district", "state"]].head(5)

    similar_list = "\n".join(
        [f"• {r['district']} ({r['state']})"
         for _, r in similar.iterrows()]
    )

    return f"""
District: {row['district']}
State: {row['state']}

Health Risk: {row['Health_Risk']}
Cluster: {row['Health_Cluster']}
Health Score: {row['Health_Score']:.1f}/100

Most Similar Districts:
{similar_list}
"""

app = gr.Interface(
    fn=analyze_district,
    inputs=gr.Dropdown(
        choices=sorted(health_df["district"].unique()),
        label="Select District"
    ),
    outputs=gr.Textbox(label="AI Analysis", lines=15),
    title="AI-Based District Health Profiling System",
    description="Select a district to analyze its health profile using K-Means clustering."
)

app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://930f63f82641849196.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
